# End to End Simple RNN Deep Learning Project Implementation

In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Input, SimpleRNN, Dense

In [2]:
## Load the imdb dataset

max_feature = 10000  # vocabulary size
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_feature)

print(f'Training data shape: {X_train.shape}, Training labels shape: {y_train.shape}')

print(f'Testing data shape: {X_test.shape}, Testing labels shape: {y_test.shape}')


Training data shape: (25000,), Training labels shape: (25000,)
Testing data shape: (25000,), Testing labels shape: (25000,)


In [3]:
## Inspect a sample review and its label

sample_review = X_train[0]
sample_label = y_train[0]

print(f'Sample review (as int): {sample_review}')
print(f'Sample label: {sample_label}')

Sample review (as int): [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]
Sample label: 1


In [4]:
len(X_train[0])

218

## Decoding of words index back to words

In [5]:
word_index = imdb.get_word_index()
# word_index

In [6]:
reverse_word_index = {value: key for key, value in word_index.items()}

# reverse_word_index

In [7]:
decoded_review = ' '.join([reverse_word_index.get(i-3, '?') for i in sample_review])
decoded_review

"? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done don't you th

In [8]:
max_len = 500

X_train = sequence.pad_sequences(X_train, maxlen=max_len)
X_test = sequence.pad_sequences(X_test, maxlen = max_len)

# X_train

In [9]:
X_train.shape

(25000, 500)

In [10]:
# X_train[0]

## Train simple RNN


In [11]:
model = Sequential()
model.add(Input(shape=(max_len,)))
model.add(Embedding(max_feature, 128))
model.add(SimpleRNN(128, activation='relu'))
model.add(Dense(1, activation='sigmoid'))



In [13]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
model.compile(optimizer='adam', loss = 'binary_crossentropy', metrics=['accuracy'])

In [18]:
## Create an instance of Earlystopping
from tensorflow.keras.callbacks import EarlyStopping
earlystoping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [ ]:
model.fit(
    X_train, y_train, epochs =10, batch_size=32,
    validation_split=0.2,
    callbacks = [earlystoping] 
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 135s 421ms/step - accuracy: 0.6873 - loss: 1.4037 - val_accuracy: 0.7182 - val_loss: 0.5266
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 126s 403ms/step - accuracy: 0.6582 - loss: 446035.0312 - val_accuracy: 0.6826 - val_loss: 0.6012
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 84s 265ms/step - accuracy: 0.7135 - loss: 0.5804 - val_accuracy: 0.6688 - val_loss: 0.5974
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 70s 223ms/step - accuracy: 0.7703 - loss: 0.5256 - val_accuracy: 0.7036 - val_loss: 0.5616
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 99s 277ms/step - accuracy: 0.8317 - loss: 0.3931 - val_accuracy: 0.7888 - val_loss: 0.4574


In [22]:
model.save('simple_rnn_imdb.h5')